# Kick-out limit animation 

Here an animation of the kick-out limit for the axion is produced. 


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from scipy.integrate import solve_ivp
import local

# --- 1. PARAMETERS SETUP AND ODE RESOLUTION ---
a = np.logspace(-6, 0, 2000)
z_arr = 1./a - 1.

logm = -30.5
tf = 100
n = 1
eps = 1e-8
z_end = 50
z_start = 60
rtol = 1e-5
atol = 1e-7
theta_ini = 0.01
bphi0_ini = theta_ini * tf
bphi1_ini = 0.

# Critical threshold and definition of the two runs
xi_crit = 1.72811e-02 # Kick limit in z=[50,60] for log_ma = -30.5
xi_safe = 0.0110      # Stable case below threshold
xi_kick = 0.0190      # Resonant case above threshold

print("Solving ODE for the Safe case (xi = 0.015)...")
phi_safe_sol = solve_ivp(local.EoM_phi, [a[0],1], [bphi0_ini,bphi1_ini], t_eval=a, 
                         args=[logm,tf,n,xi_safe,eps,z_start,z_end, bphi0_ini], 
                         atol=atol, rtol=rtol, method="RK45")
sol_safe = phi_safe_sol['y'][0]

print("Solving ODE for the Kick-out case (xi = 0.019)...")
phi_kick_sol = solve_ivp(local.EoM_phi, [a[0],1], [bphi0_ini,bphi1_ini], t_eval=a, 
                         args=[logm,tf,n,xi_kick,eps,z_start,z_end, bphi0_ini], 
                         atol=atol, rtol=rtol, method="RK45")
sol_kick = phi_kick_sol['y'][0]


# --- 2. FIGURE SETUP ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Panel 1: Normalized Potential
# The cosine well goes from -pi*tf to +pi*tf (roughly -314 to +314)
def V_full(phi):
    return 1 - np.cos(phi / tf)

phi_grid = np.linspace(-400, 400, 1000)
ax1.plot(phi_grid, V_full(phi_grid), color='gray', linestyle='--', alpha=0.7)

# Proper framing to see the field and the left wall of the potential
ax1.set_xlim(-350, 200) 
ax1.set_ylim(-0.1, 2.2)
ax1.set_xlabel(r"$\phi$ field", fontsize=14)
ax1.set_ylabel(r"Normalized Potential $V(\phi)$", fontsize=14)
ax1.axhline(0, color='k', linewidth=0.5)

ball_safe, = ax1.plot([], [], 'o', color='C0', markersize=10, label=r"Safe ($\xi = 0.015$)")
ball_kick, = ax1.plot([], [], 'o', color='C3', markersize=10, label=r"Kick-out ($\xi = 0.019$)")
ax1.legend(loc="upper right", fontsize=12)

# Panel 2: Field temporal dynamics
ax2.set_xscale('log')
ax2.set_xlim(0.1, 2.2e3) # Decreasing order: from right (2200) to left (0.01)

# CORRECT Y SCALE AS REQUESTED
ax2.set_ylim(-300, 150)

ax2.set_xlabel(r'Redshift $z$', fontsize=14)
ax2.set_ylabel(r'$\phi(z)$', fontsize=14) # Or add [deg] if you already multiplied by beta0

# Colored zones
ax2.axvspan(z_start, z_end, alpha=0.1, color='red', label='Injection zone')
ax2.axvspan(10, 5, alpha=0.1, color='blue', label='Reionization epoch')
ax2.axhline(0, color='k', ls='--', lw=0.5)

line_safe, = ax2.plot([], [], '-', color='C0', lw=2.0, label='Stable evolution')
line_kick, = ax2.plot([], [], '--', color='C3', lw=2.0, label='Kick-out trj')
ax2.legend(loc="upper left", fontsize=12)
ax2.grid(axis='y', linestyle='--', alpha=0.6)
ax2.grid(axis='x', linestyle=':', alpha=0.4)

plt.tight_layout()

# --- 3. ANIMATION ---
def init():
    ball_safe.set_data([], [])
    ball_kick.set_data([], [])
    line_safe.set_data([], [])
    line_kick.set_data([], [])
    return ball_safe, ball_kick, line_safe, line_kick

# Step > 1 to speed up animation generation
step = 5 
total_frames = len(a) // step

def update(f):
    idx = f * step
    if idx >= len(a): idx = len(a) - 1
    
    # Protection in case the solver stopped early in the kick-out run
    idx_safe = min(idx, len(sol_safe) - 1)
    idx_kick = min(idx, len(sol_kick) - 1)
    
    x_safe = sol_safe[idx_safe]
    x_kick = sol_kick[idx_kick]
    
    # Update balls position
    ball_safe.set_data([x_safe], [V_full(x_safe)])
    
    # If the red ball goes beyond -350, we hide it for visual clarity
    if x_kick > -350:
        ball_kick.set_data([x_kick], [V_full(x_kick)])
        ball_kick.set_alpha(1)
    else:
        ball_kick.set_alpha(0)
        
    # Draw the history line
    line_safe.set_data(z_arr[:idx_safe], sol_safe[:idx_safe])
    line_kick.set_data(z_arr[:idx_kick], sol_kick[:idx_kick])
    
    return ball_safe, ball_kick, line_safe, line_kick

ani = animation.FuncAnimation(fig, update, frames=total_frames, init_func=init, blit=True, interval=25)

# --- 4. OUTPUT ---
print("Generating the animation... Please wait.")
plt.close()

# Save the animation as a video
ani.save("kickout_real.mp4", fps=30, dpi=150, extra_args=['-vcodec', 'libx264'])
print("Animation completed and saved!")

Solving ODE for the Safe case (xi = 0.015)...
Solving ODE for the Kick-out case (xi = 0.019)...
Generating the animation... Please wait.
Animation completed and saved!
